# FRB population DM distributions: all surveys, one set of parameters

Shared physics lives in `core.py`, shared parameters in `params.py`, per-telescope details in `telescopes.py` / `beams.py`.
Tune the parameters in the next cell **once**; every survey picks them up.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt
from params import SimParams
from telescopes import ALL_KEYS, OUT_DIR, get_telescope
from core import run_survey

In [ ]:
# ---- the shared knobs (previously copy-pasted into 9 notebooks) ---------------------------------
p = SimParams(
    alpha_intrinsic=0,
    USE_COSMOLOGY=True, USE_SFR=True, USE_SMEARING=True, USE_SPECTRAL_INDEX=True, USE_BEAM=True,
    LUMINOSITY_FUNCTION="schechter",          # or "std"
    L_STAR=3e13, ALPHA_SCH=-1.2, L_MIN_SCH=1e6, L_MAX_SCH=1e16,
    L_std=1e12,
    fscale=100e-7, SNR_threshold=8, t_obs=0.001,
)
TAG = ""            # e.g. "_snr9" so a trial run does not overwrite your reference .npy files
KEYS = ALL_KEYS     # or e.g. ["pks", "fast"]
print(KEYS)

In [ ]:
results = {}
for key in KEYS:
    results[key] = run_survey(get_telescope(key), p)
    print("  saved ->", results[key].save(OUT_DIR, TAG))

In [ ]:
# quick look: weighted DM histogram of every survey
fig, axes = plt.subplots(3, 3, figsize=(15, 11), dpi=100, sharex=True)
for ax, (key, res) in zip(axes.flat, results.items()):
    arr = res.dm_distribution
    ax.hist(arr[:, 0], bins=np.linspace(arr[:, 0].min(), arr[:, 0].max(), 40), weights=arr[:, 1],
            density=True, alpha=0.6, color=get_telescope(key).color, label=f"{res.label}, n$_{{frb}}$={int(res.n_frb)}")
    ax.legend(fontsize=9)
for ax in axes[-1]: ax.set_xlabel("DM (pc/cm$^3$)")
plt.tight_layout(); plt.show()

## Parameter sweep example
Same machinery, one parameter changed, nothing overwritten (note the `TAG`):

In [ ]:
# for a in (-1.0, -1.2, -1.5):
#     pa = p.with_overrides(ALPHA_SCH=a)
#     for key in KEYS:
#         run_survey(get_telescope(key), pa, verbose=False).save(OUT_DIR, f"_alpha{a}")